In [2]:
from xbbg import blp
import pandas as pd
from datetime import datetime, timedelta

async def get_earnings_calendar():
    try:
        print("Fetching S&P 500 members...")
        members = await blp.abds('SPX Index', 'INDX_MEMBERS')
        ticker_col = 'Member Ticker and Exchange Code'
        tickers = [f"{m} Equity" for m in members[ticker_col]]
        
        fields = ['EXPECTED_REPORT_DT', 'EARN_ANN_DT', 'SHORT_NAME']
        
        print(f"Fetching dates for {len(tickers)} tickers...")
        data = await blp.abdp(tickers, fields)
        
        df = data.to_pandas()
        df_pivot = df.pivot(index='ticker', columns='field', values='value')
        
        date_col = 'EXPECTED_REPORT_DT' if 'EXPECTED_REPORT_DT' in df_pivot.columns else 'EARN_ANN_DT'
        df_pivot[date_col] = pd.to_datetime(df_pivot[date_col], errors='coerce')
        
        today = datetime.now()
        next_week = today + timedelta(days=7)
        
        calendar = df_pivot[
            (df_pivot[date_col] >= today.strftime('%Y-%m-%d')) & 
            (df_pivot[date_col] <= next_week.strftime('%Y-%m-%d'))
        ].sort_values(date_col)
        
        return calendar, date_col

    except Exception as e:
        print(f"Error: {e}")
        return None, None

# Run and display all rows
df_calendar, date_field = await get_earnings_calendar()


Fetching S&P 500 members...


Fetching dates for 503 tickers...


# Earnings Calendar

In [3]:
if df_calendar is not None and not df_calendar.empty:
    print(f"\nEarnings Calendar (Next 7 Days - {len(df_calendar)} companies):")
    # Set options to show all rows
    with pd.option_context('display.max_rows', None):
        display(df_calendar[[date_field, 'SHORT_NAME']])
else:
    print("No earnings found.")


Earnings Calendar (Next 7 Days - 18 companies):


field,EXPECTED_REPORT_DT,SHORT_NAME
ticker,,
AMAT UW Equity,2026-05-14,APPLIED MATERIAL
HD UN Equity,2026-05-19,HOME DEPOT INC
KEYS UN Equity,2026-05-19,KEYSIGHT TEC
ADI UW Equity,2026-05-20,ANALOG DEVICES
TJX UN Equity,2026-05-20,TJX COS INC
TGT UN Equity,2026-05-20,TARGET CORP
NVDA UW Equity,2026-05-20,NVIDIA CORP
NDSN UW Equity,2026-05-20,NORDSON CORP
LOW UN Equity,2026-05-20,LOWE'S COS INC
